# Exploratory SAR corridor changes and reference checks

Three pre-event dates and one post-event date, same track/polarizations, explicit aligned grids, linear-power smoothing and compositing. This notebook does not classify floodwater or assign institution risk labels.

Cached chips support offline analysis. Enable the download cell only for an intentional new raw snapshot after authenticating. Change experiment settings only alongside the implementation and a new dataset ID/source directory. Reruns replace derived products and record a new run.

In [ ]:
from pathlib import Path
import json

# In Colab, clone the repository and change into it before running this cell.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/ethekwini.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the africa-map checkout or one of its subdirectories.")
CONFIG = json.loads((ROOT / "config/ethekwini.yaml").read_text())
print("Project:", CONFIG["project_id"])


In [ ]:
RUN_REMOTE_DOWNLOADS = False
if RUN_REMOTE_DOWNLOADS:
    from pathlib import Path
    from urllib.request import urlopen
    from datetime import datetime,timezone,timedelta
    import json,math
    import geopandas as gpd
    import ee
    root=ROOT;cfg=CONFIG;ee.Initialize(project=cfg['availability']['earth_engine_project'])
    raw=root/'data/raw/sar_corridor_review';raw.mkdir(parents=True,exist_ok=True)
    sites=gpd.read_file(root/'data/processed/ethekwini/flagged_site_review.gpkg',layer='osm_outlines').to_crs(32736)
    previous=json.loads((raw/'download_manifest.json').read_text()) if (raw/'download_manifest.json').exists() else []
    manifest=[]
    for site in sites.itertuples():
     b=site.geometry.buffer(1000).bounds;bounds=[math.floor(b[0]/10)*10,math.floor(b[1]/10)*10,math.ceil(b[2]/10)*10,math.ceil(b[3]/10)*10]
     width,height=round((bounds[2]-bounds[0])/10),round((bounds[3]-bounds[1])/10)
     transform=[10,0,bounds[0],0,-10,bounds[3]]
     region=ee.Geometry.Rectangle(bounds,proj='EPSG:32736',geodesic=False)
     center=ee.Geometry.Point(list(site.geometry.centroid.coords)[0],proj='EPSG:32736')
     for date in ['2022-03-12','2022-03-24','2022-04-05','2022-04-17']:
      end=(datetime.fromisoformat(date)+timedelta(days=1)).date().isoformat()
      coll=ee.ImageCollection('COPERNICUS/S1_GRD').filterDate(date,end).filterBounds(center).filter(ee.Filter.eq('relativeOrbitNumber_start',43)).filter(ee.Filter.eq('orbitProperties_pass','ASCENDING')).filter(ee.Filter.eq('instrumentMode','IW')).filter(ee.Filter.listContains('transmitterReceiverPolarisation','VV')).filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH'))
      info=coll.map(lambda x:ee.Feature(None,{'asset_id':ee.String('COPERNICUS/S1_GRD/').cat(ee.String(x.get('system:index'))),'acquired_ms':x.get('system:time_start'),'orbit':x.get('relativeOrbitNumber_start'),'pass':x.get('orbitProperties_pass'),'polarizations':x.get('transmitterReceiverPolarisation')})).getInfo()['features']
      assert len(info)==1,info
      meta=info[0]['properties'];name=f'{site.institution_id.split("/")[-1]}_{date}.tif';path=raw/name
      if not path.exists():
       image=ee.Image(meta['asset_id']).select(['VV','VH','angle']).toFloat()
       url=image.getDownloadURL({'crs':'EPSG:32736','crs_transform':transform,'dimensions':[width,height],'format':'GEO_TIFF','filePerBand':False})
       with urlopen(url,timeout=60) as response:data=response.read()
       path.write_bytes(data)
      manifest.append({'file':name,'institution_id':site.institution_id,'date':date,'role':'post_event' if date=='2022-04-17' else 'pre_event','bands':['VV','VH','angle'],'crs':'EPSG:32736','transform':transform,'width':width,'height':height,'bounds':bounds,'asset':meta,'downloaded_at_utc':next((r['downloaded_at_utc'] for r in previous if r['file']==name),datetime.now(timezone.utc).isoformat())})
      (raw/'download_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n');print(name,path.stat().st_size,flush=True)
else:
    print("Using retained aligned chips; no remote calls.")


In [ ]:
assert CONFIG['sar_change'] == {'source_directory':'data/raw/sar_corridor_review','pre_dates':['2022-03-12','2022-03-24','2022-04-05'],'post_date':'2022-04-17','relative_orbit':43,'orbit_state':'ASCENDING','region_buffer_m':1000,'grid_scale_m':10,'filter_window_pixels':3,'corridor_buffer_m':250,'reference_exclusion_m':500,'reference_building_exclusion_m':100,'reference_march_pair_max_spread_db':1.0,'decrease_thresholds_db':[1.5,3.0,4.5]}, 'Review analysis implementation and register a new experiment before changing these settings.'
from pathlib import Path
import os
os.environ.setdefault('MPLCONFIGDIR','/tmp/africa-map-matplotlib')
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from heritage_flood.sar_change import analyze_sar

raw=ROOT/'data/raw/sar_corridor_review';manifest=json.loads((raw/'download_manifest.json').read_text())
output=ROOT/CONFIG['output_directory'];figures=ROOT/'outputs/figures';figures.mkdir(parents=True,exist_ok=True)
sites=gpd.read_file(output/'flagged_site_review.gpkg',layer='osm_outlines').to_crs(32736)
flood=gpd.read_file(output/'floodplain_screening.gpkg',layer='mapped_floodplain').to_crs(32736).geometry.union_all()
rows=[];checks=[]
for site in sites.itertuples():
 id=site.institution_id.split('/')[-1];entries=sorted([r for r in manifest if r['institution_id']==site.institution_id],key=lambda r:r['date'])
 assert [r['date'] for r in entries]==['2022-03-12','2022-03-24','2022-04-05','2022-04-17']
 arrays=[];angles=[];reference=None
 for item in entries:
  assert item['asset']['orbit']==43 and item['asset']['pass']=='ASCENDING' and set(item['asset']['polarizations'])=={'VV','VH'}
  with rasterio.open(raw/item['file']) as image:
   profile=image.profile.copy()
   if reference is None:reference=(image.transform,image.crs,image.shape);extent=[image.bounds.left,image.bounds.right,image.bounds.bottom,image.bounds.top]
   assert (image.transform,image.crs,image.shape)==reference
   assert image.crs.to_epsg()==32736 and image.res==(10,10)
   data=image.read(masked=True).astype(float).filled(np.nan)
   # Missing incidence marks unsupported pixels in this local product, not dry ground.
   data[:,~np.isfinite(data[2]) | (data[2]<=0)]=np.nan
   arrays.append(data[:2]);angles.append(data[2])
 pre=np.stack(arrays[:3]);post=arrays[3];result=analyze_sar(pre,post,window=3)
 shape=post.shape[1:];transform=reference[0]
 inside=geometry_mask([flood.__geo_interface__],out_shape=shape,transform=transform,invert=True)
 corridor=geometry_mask([flood.buffer(250).__geo_interface__],out_shape=shape,transform=transform,invert=True)
 far=~geometry_mask([flood.buffer(500).__geo_interface__],out_shape=shape,transform=transform,invert=True)
 away_from_building=~geometry_mask([site.geometry.buffer(100).__geo_interface__],out_shape=shape,transform=transform,invert=True)
 valid=np.isfinite(result['delta_db']).all(axis=0)&np.isfinite(result['holdout_delta_db']).all(axis=0)
 references=far & away_from_building & (result['reference_pair_spread_db']<=1.0) & valid
 assert not (references & corridor).any()
 regions={'all_context':valid,'mapped_polygon':inside & valid,'river_corridor_250m':corridor & valid,'preselected_references':references}
 for region,mask in regions.items():
  n=int(mask.sum())
  for comparison,delta in [('post_vs_3pre',result['delta_db']),('heldout_pre_vs_2pre',result['holdout_delta_db']),('post_vs_2pre',result['post_vs_training_delta_db'])]:
   for threshold in [1.5,3,4.5]:
    count=int(((delta[0]<=-threshold)&(delta[1]<=-threshold)&mask).sum())
    rows.append({'institution_id':site.institution_id,'region':region,'comparison':comparison,'decrease_threshold_db':threshold,'valid_grid_centers':n,'joint_decrease_grid_centers':count,'joint_decrease_fraction':count/n if n else None,'joint_decrease_sampled_area_m2':count*100 if n else None,'median_vv_delta_db':float(np.median(delta[0][mask])) if n else None,'median_vh_delta_db':float(np.median(delta[1][mask])) if n else None,'interpretation':'backscatter decrease only; not a flood or dry label'})
 checks.append({'institution_id':site.institution_id,'valid_grid_centers':int(valid.sum()),'reference_grid_centers':int(references.sum()),'reference_selection_dates':['2022-03-12','2022-03-24'],'heldout_date':'2022-04-05','post_date':'2022-04-17','incidence_angle_medians_deg':[float(np.nanmedian(x[valid])) for x in angles]})
 bands=np.concatenate([result['baseline_db'],result['post_db'],result['delta_db'],result['holdout_delta_db'],result['post_vs_training_delta_db'],result['reference_pair_spread_db'][None],corridor[None].astype(float),references[None].astype(float),valid[None].astype(float)])
 names=['baseline_vv_db','baseline_vh_db','post_vv_db','post_vh_db','delta_vv_db','delta_vh_db','heldout_delta_vv_db','heldout_delta_vh_db','post_training_delta_vv_db','post_training_delta_vh_db','reference_pair_spread_db','corridor_250m','reference_selected','joint_valid']
 profile.update(count=len(names),dtype='float32',nodata=np.nan,compress='deflate')
 with rasterio.open(output/f'sar_change_{id}.tif','w',**profile) as dest:
  dest.write(bands.astype('float32'));dest.descriptions=tuple(names)
  dest.update_tags(interpretation='EXPLORATORY_BACKSCATTER_CHANGE_NOT_FLOOD_CLASSIFICATION',baseline_dates='2022-03-12;2022-03-24;2022-04-05',post_date='2022-04-17',filter='3x3 mean in linear power; complete support only')
 fig,axes=plt.subplots(2,3,figsize=(13,9),layout='constrained')
 panels=[(result['baseline_db'][0],'Three-date power baseline VV',-25,0,'gray'),(result['post_db'][0],'17 April VV, spatially smoothed',-25,0,'gray'),(result['delta_db'][0],'VV change: post minus three-date baseline',-6,6,'RdBu'),(result['delta_db'][1],'VH change: post minus three-date baseline',-6,6,'RdBu'),(result['holdout_delta_db'][0],'Withheld 5 April VV minus March baseline',-6,6,'RdBu')]
 for ax,(data,title,lo,hi,cmap) in zip(axes.flat,panels):
  im=ax.imshow(data,extent=extent,vmin=lo,vmax=hi,cmap=cmap,interpolation='nearest')
  fig.colorbar(im,ax=ax,shrink=.7,label='dB')
  gpd.GeoSeries([site.geometry],crs=32736).boundary.plot(ax=ax,color='#00ffff',linewidth=1)
  ax.set_title(title,fontsize=9)
 ax=axes.flat[-1]
 classes=np.zeros(shape);classes[corridor]=1;classes[references]=2;classes[~valid]=np.nan
 from matplotlib.colors import ListedColormap
 ax.imshow(classes,extent=extent,cmap=ListedColormap(['#f3f4f6','#fed7aa','#15803d']),vmin=0,vmax=2,interpolation='nearest')
 gpd.GeoSeries([site.geometry],crs=32736).boundary.plot(ax=ax,color='#0891b2',linewidth=1)
 ax.set_title('Orange: mapped-polygon corridor (+250 m)\nGreen: preselected reference candidates',fontsize=9)
 for ax in axes.flat:
  ax.set_xlim(extent[:2]);ax.set_ylim(extent[2:]);ax.set_aspect('equal');ax.set_xticks([]);ax.set_yticks([]);ax.set_xlabel('');ax.set_ylabel('')
  ax.plot([extent[0]+70,extent[0]+570],[extent[2]+80,extent[2]+80],color='#111827',linewidth=2)
  ax.text(extent[0]+320,extent[2]+105,'500 m',ha='center',fontsize=8,bbox={'facecolor':'white','alpha':.85,'edgecolor':'none','pad':1})
  ax.text(.96,.96,'N ↑',ha='right',va='top',transform=ax.transAxes,fontsize=9,bbox={'facecolor':'white','alpha':.85,'edgecolor':'none','pad':1})
 name='Unnamed library (candidate Amanzimtoti Library)' if id=='368501740' else 'Umkhumbane Museum'
 fig.suptitle(name+' — exploratory SAR change\n10 m grid, 3×3 linear-power smoothing; cyan = current OSM outline',fontsize=12)
 fig.supxlabel('Negative/red = lower backscatter; positive/blue = higher backscatter. Neither is a flood label.\nSentinel: EU/ESA/Copernicus via Earth Engine; OSM: © OpenStreetMap contributors; municipal corridor is historical mapping.',fontsize=8)
 fig.savefig(figures/f'sar_change_{id}.png',dpi=150);plt.close(fig)
pd.DataFrame(rows).to_csv(output/'sar_change_sensitivity.csv',index=False)
(ROOT/'metadata/sar_change_checks.json').write_text(json.dumps(checks,indent=2)+'\n')
print('Exploratory change rasters, figures and sensitivity table written.')


In [ ]:
from datetime import datetime, timezone
import csv
import hashlib
import platform
import shutil
import subprocess

started=datetime.now(timezone.utc)
run_id="sar_change_"+started.strftime("%Y%m%dT%H%M%S%fZ")
summary={"run_id":run_id,"parameters":CONFIG["sar_change"],"sites":checks,
    "baseline_method":"3x3 spatial mean in linear power per acquisition, then temporal mean power of all three pre-event dates, converted to dB",
    "reference_selection":"Outside mapped polygon +500 m and building +100 m; March pair spread <=1 dB in both channels; valid in all compared scenes",
    "comparison_guard":"Withheld April 5 and post-event April 17 are separately compared to the same March 12/24 mean power for like-baseline diagnostics",
    "finding":"Joint VV/VH decrease fractions are lower post-event than in the withheld pre-event corridor comparison at all tested thresholds; no distinctive event-flood signal established by this method",
    "flood_labels_assigned":False,"limitations":"Only three pre-event dates, single late post-event date, empirical reference selection, no independent flood validation, no land-cover/terrain/permanent-water masks; spatial smoothing correlates samples"}
summary_path=ROOT/"metadata/sar_change_summary.json"
summary_path.write_text(json.dumps(summary,indent=2)+"\n")
run_directory=ROOT/"metadata/runs"/run_id;run_directory.mkdir(parents=True,exist_ok=False)
(run_directory/"config.json").write_text(json.dumps(CONFIG,indent=2)+"\n")
env={"python":platform.python_version(),"numpy":np.__version__,"rasterio":rasterio.__version__,"geopandas":gpd.__version__,"matplotlib":matplotlib.__version__}
(run_directory/"environment.json").write_text(json.dumps(env,indent=2)+"\n")
source_files=[raw/"download_manifest.json"]+[raw/r["file"] for r in manifest]
source_manifest=ROOT/"metadata/sar_corridor_source_checksums.json"
source_manifest.write_text(json.dumps({str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files},indent=2)+"\n")
inputs=source_files+[output/"flagged_site_review.gpkg",output/"floodplain_screening.gpkg"]
(run_directory/"input_checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs},indent=2)+"\n")
code=[ROOT/"src/heritage_flood/sar_change.py",ROOT/"notebooks/06_sar_change.ipynb"]
for p in code:shutil.copyfile(p,run_directory/p.name)
(run_directory/"code_checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in code},indent=2)+"\n")
outputs=[output/f"sar_change_{way}.tif" for way in CONFIG["site_review"]["way_ids"]]+[figures/f"sar_change_{way}.png" for way in CONFIG["site_review"]["way_ids"]]+[output/"sar_change_sensitivity.csv",ROOT/"metadata/sar_change_checks.json",summary_path,source_manifest]
(run_directory/"checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in outputs},indent=2)+"\n")
dataset_id="sar_corridor_chips_20261020"
source_row={"dataset_id":dataset_id,"title":"Aligned Sentinel-1 corridor cutouts, three pre-event and one post-event dates","provider":"EU/ESA/Copernicus via Google Earth Engine","version":"2026-10-20 downloads; exact assets and aligned grids retained","source_url_or_asset_id":"COPERNICUS/S1_GRD","status":"accessed","accessed_at_utc":min(r["downloaded_at_utc"] for r in manifest),"observation_start":"2022-03-12","observation_end":"2022-04-17","geographic_coverage":"Two approximately 2 km local contexts around flagged institutions","resolution_or_scale":"10 m sampling; 3x3 analysis smoothing does not improve spatial resolution","crs":"EPSG:32736","license":"Copernicus Sentinel data terms; municipal/OSM references retain separate conditions","redistribution_conditions":"Local exploratory analysis; nothing published to website","limitations":summary["limitations"]+" Register sha256 hashes the source-file checksum manifest.","storage_location":str(source_manifest.relative_to(ROOT)),"sha256":hashlib.sha256(source_manifest.read_bytes()).hexdigest()}
run_row={"run_id":run_id,"started_at_utc":started.isoformat(),"input_dataset_ids":";".join([dataset_id,"osm_flagged_sites_snapshot_20261020",CONFIG["floodplain_dataset_id"]]),"scene_manifest_path":str((raw/"download_manifest.json").relative_to(ROOT)),"code_commit":subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip(),"working_tree_dirty":str(bool(subprocess.check_output(["git","status","--porcelain"],cwd=ROOT,text=True).strip())).lower(),"config_snapshot_path":str((run_directory/"config.json").relative_to(ROOT)),"environment_path":str((run_directory/"environment.json").relative_to(ROOT)),"method":"Aligned matched-track SAR power-domain smoothing/compositing; March-only reference selector; April 5 holdout; post-minus-baseline changes and joint-decrease sensitivity","output_locations":";".join(str(p.relative_to(ROOT)) for p in outputs),"output_checksums_path":str((run_directory/"checksums.json").relative_to(ROOT)),"validation_status":"partial","notes":"Numerical and alignment checks passed; figures inspected; no independent flood validation or flood labels. See research/sar_change.md."}
for filename,key,row in [("datasets.csv","dataset_id",source_row),("processing_runs.csv","run_id",run_row)]:
    path=ROOT/"metadata"/filename
    with path.open(newline="") as handle:
        reader=csv.DictReader(handle);fields,existing=reader.fieldnames,list(reader)
    previous=next((r for r in existing if r[key]==row[key]),None)
    if filename=="datasets.csv" and previous and previous["sha256"]!=row["sha256"]:raise ValueError("Raw snapshot changed; assign a new dataset ID")
    with path.open("w",newline="") as handle:
        writer=csv.DictWriter(handle,fieldnames=fields,lineterminator="\n");writer.writeheader();writer.writerows([r for r in existing if r[key]!=row[key]]+[row])
print("Registered exploratory SAR run:",run_id)


## Interpretation

Read `research/sar_change.md`. Reference candidates use March-pair stability and geographic exclusion only; they are not independently verified stable/unflooded controls. The April 5 holdout is not used for reference selection. Compare holdout and post-event against the same March baseline when interpreting background variation.

The primary map uses all three pre-event dates. Decreases and threshold fractions are backscatter diagnostics, not inundation labels. Filtering correlates neighboring samples; pixel counts are not independent statistical trials. View the joint_valid raster band before interpreting any output. The April 17 observation does not rule out earlier transient inundation.